# Analyse exploratoire (EDA) : PlantDoc embarqué sur Android

Ce notebook analyse **les données**, **l'entraînement** et **les performances des 4 variantes** du modèle (FP32, FP16, INT8 dynamique, INT8 statique).

**Comment l'utiliser**
1. Ouvrir ce fichier dans VS Code, Jupyter ou JupyterLab, avec l'environnement `.venv` du projet (`pip install jupyter` si besoin).
2. Menu **Run All** (exécuter toutes les cellules). L'évaluation des 4 modèles prend 1 à 2 minutes.
3. Le notebook doit être ouvert depuis le projet : il retrouve tout seul la racine (dossier contenant `README.md`).

Les graphiques s'affichent dans le notebook ; rien n'est écrit sur le disque.

## 0. Préparation

In [ ]:
import os
import re
import sys
import time
from pathlib import Path

# Se placer à la racine du projet (dossier qui contient README.md) et rendre les scripts importables
root = Path.cwd()
while not (root / "README.md").exists() and root.parent != root:
    root = root.parent
os.chdir(root)
sys.path.insert(0, str(root / "scripts"))
print("Racine du projet :", root)

import matplotlib.pyplot as plt
import numpy as np
import onnxruntime as ort
import pandas as pd
from IPython.display import Markdown, display
from PIL import Image, ImageDraw

import plantdoc_pipeline as P   # pré/post-traitement et calcul d'AP déjà utilisés pour le benchmark

DATA = Path("data/plantdoc_yolo")
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)

In [ ]:
# Palette de référence (couleurs catégorielles + neutres) et style des graphiques
SURFACE, INK, INK2, MUTED, GRID, AXIS = "#fcfcfb", "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7"
BLUE, ORANGE, AQUA = "#2a78d6", "#eb6834", "#1baf7a"
VARIANT_COLOR = {"fp32": MUTED, "fp16": AQUA, "int8_dynamic": ORANGE, "int8_static": BLUE}
VARIANT_LABEL = {"fp32": "FP32", "fp16": "FP16", "int8_dynamic": "INT8 dynamique", "int8_static": "INT8 statique"}
TYPE_COLOR = {"saine": AQUA, "malade": ORANGE}

plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": AXIS, "axes.labelcolor": INK2, "xtick.color": MUTED, "ytick.color": MUTED,
    "text.color": INK, "axes.titlecolor": INK, "axes.titlesize": 12, "axes.titleweight": "bold",
    "axes.titlelocation": "left", "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8, "axes.axisbelow": True,
    "font.family": "sans-serif", "font.size": 10, "figure.dpi": 100,
})

## 1. Chargement des données
On lit les étiquettes YOLO des trois découpes (train, val, test) : une ligne par boîte et une ligne par image.

In [ ]:
names = [re.match(r"^\s+\d+:\s*(.*)$", l).group(1)
         for l in (DATA / "data.yaml").read_text(encoding="utf-8").splitlines() if re.match(r"^\s+\d+:", l)]
healthy = [l.startswith("H|") for l in Path("android/app/src/main/assets/classes.txt").read_text(encoding="utf-8").splitlines() if l.strip()]
NC = len(names)

rows, imgs = [], []
for split in ("train", "val", "test"):
    for lab in sorted((DATA / "labels" / split).glob("*.txt")):
        w, h = Image.open(DATA / "images" / split / (lab.stem + ".jpg")).size
        boxes = [l.split() for l in lab.read_text().splitlines() if l.strip()]
        imgs.append(dict(split=split, stem=lab.stem, w=w, h=h, n_boxes=len(boxes)))
        for c, cx, cy, bw, bh in boxes:
            rows.append(dict(split=split, stem=lab.stem, cls=int(c), cx=float(cx), cy=float(cy), bw=float(bw), bh=float(bh)))
B, I = pd.DataFrame(rows), pd.DataFrame(imgs)
B["area"] = B.bw * B.bh
B["type"] = ["saine" if healthy[c] else "malade" for c in B.cls]
print(f"{len(I)} images, {len(B)} boîtes, {NC} classes ({sum(healthy)} saines, {NC - sum(healthy)} maladies ou parasites)")

## 2. Les données
### Découpes

In [ ]:
split_tbl = pd.DataFrame({
    "Images": I.groupby("split").size(),
    "Boîtes": B.groupby("split").size(),
    "Boîtes / image": (B.groupby("split").size() / I.groupby("split").size()).round(2),
    "Classes présentes": B.groupby("split").cls.nunique(),
}).loc[["train", "val", "test"]]
split_tbl.loc["total"] = [len(I), len(B), round(len(B) / len(I), 2), B.cls.nunique()]
split_tbl

### Boîtes par classe

In [ ]:
cls_counts = B.pivot_table(index="cls", columns="split", values="stem", aggfunc="count", fill_value=0).reindex(range(NC), fill_value=0)
cls_counts = cls_counts[["train", "val", "test"]]
cls_counts["total"] = cls_counts.sum(axis=1)

order = cls_counts.total.sort_values().index
fig, ax = plt.subplots(figsize=(8.5, 8.2))
for k, c in enumerate(order):
    t = "saine" if healthy[c] else "malade"
    ax.barh(k, cls_counts.total[c], color=TYPE_COLOR[t], height=0.62)
    ax.text(cls_counts.total[c] + 8, k, str(cls_counts.total[c]), va="center", fontsize=8.5, color=INK2)
ax.set_yticks(range(NC)); ax.set_yticklabels([names[c] for c in order], fontsize=8.5, color=INK2)
ax.set_xlabel("Nombre de boîtes (train + val + test)"); ax.grid(axis="y", visible=False)
ax.set_title("Répartition des boîtes par classe")
ax.legend([plt.Rectangle((0, 0), 1, 1, color=TYPE_COLOR[t]) for t in ("saine", "malade")],
          ["Feuille saine", "Maladie ou parasite"], frameon=False, loc="lower right", labelcolor=INK2)
plt.show()

ratio = cls_counts.total.max() / max(cls_counts.total.min(), 1)
top5 = cls_counts.total.sort_values(ascending=False).head(5).sum() / len(B) * 100
display(Markdown(f"Classe la plus fréquente : **{int(cls_counts.total.max())}** boîtes ; la moins fréquente : **{int(cls_counts.total.min())}** "
                 f"(rapport {ratio:.0f}:1). Les 5 classes les plus fréquentes représentent **{top5:.0f} %** des boîtes."))

### Composition : images, boîtes par image, taille des boîtes

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(13, 3.9))
ax = axs[0]
sp = ["train", "val", "test"]
ax.bar(sp, [split_tbl.loc[s, "Images"] for s in sp], color=[BLUE, ORANGE, AQUA], width=0.55)
for k, s in enumerate(sp):
    ax.text(k, split_tbl.loc[s, "Images"], f"{int(split_tbl.loc[s, 'Images'])}", ha="center", va="bottom", fontsize=9, color=INK2)
ax.set_title("Images par découpe"); ax.grid(axis="x", visible=False)
ax = axs[1]
ax.hist(I.n_boxes.clip(upper=10), bins=np.arange(0.5, 11.5, 1), color=BLUE, rwidth=0.78)
ax.set_title("Boîtes par image"); ax.set_xlabel("Boîtes (10 = 10 ou plus)"); ax.set_xticks(range(1, 11)); ax.grid(axis="x", visible=False)
ax = axs[2]
ax.hist(B.area * 100, bins=np.linspace(0, 100, 26), color=BLUE, rwidth=0.82)
med = B.area.median() * 100
ax.axvline(med, color=ORANGE, lw=2); ax.text(med + 1.5, ax.get_ylim()[1] * 0.9, f"médiane {med:.0f} %", color=INK2, fontsize=9)
ax.set_title("Surface d'une boîte (% de l'image)"); ax.set_xlabel("Surface relative (%)"); ax.grid(axis="x", visible=False)
plt.show()

### Exemples annotés (jeu de test)

In [ ]:
picks, seen = [], set()
for lab in sorted((DATA / "labels" / "test").glob("*.txt")):
    cs = {int(l.split()[0]) for l in lab.read_text().splitlines() if l.strip()}
    if len(cs) == 1 and cs.isdisjoint(seen) and len(picks) < 6:
        seen |= cs
        picks.append(lab)
fig, axs = plt.subplots(2, 3, figsize=(11, 7.4))
for ax, lab in zip(axs.ravel(), picks):
    im = Image.open(DATA / "images" / "test" / (lab.stem + ".jpg")).convert("RGB")
    d = ImageDraw.Draw(im)
    for l in lab.read_text().splitlines():
        c, cx, cy, bw, bh = l.split(); c = int(c); cx, cy, bw, bh = [float(v) for v in (cx, cy, bw, bh)]
        col = AQUA if healthy[c] else ORANGE
        d.rectangle([(cx - bw / 2) * im.width, (cy - bh / 2) * im.height, (cx + bw / 2) * im.width, (cy + bh / 2) * im.height], outline=col, width=4)
    ax.imshow(im); ax.axis("off"); ax.set_title(names[c], fontsize=10)
fig.suptitle("Boîtes annotées (turquoise = saine, orange = malade)", x=0.01, ha="left", fontsize=11, fontweight="bold")
plt.show()

## 3. L'entraînement
Courbes des pertes et du score sur la validation, par epoch. On prend l'entraînement le plus complet trouvé dans `runs/`.

In [ ]:
cands = sorted(Path("runs").rglob("plantdoc/train*/results.csv"), key=lambda q: len(q.read_text().splitlines()))
res_csv = cands[-1] if cands else None
train_note = None
if res_csv is None:
    print("Aucun journal d'entraînement trouvé dans runs/ : cette section est ignorée.")
else:
    R = pd.read_csv(res_csv); R.columns = [c.strip() for c in R.columns]
    best = int(R["metrics/mAP50(B)"].idxmax())
    fig, axs = plt.subplots(1, 2, figsize=(12, 3.9))
    ax = axs[0]
    ax.plot(R.epoch, R["train/box_loss"], color=BLUE, lw=2, label="box")
    ax.plot(R.epoch, R["train/cls_loss"], color=ORANGE, lw=2, label="classification")
    ax.set_title("Pertes d'entraînement"); ax.set_xlabel("Epoch"); ax.legend(frameon=False, labelcolor=INK2)
    ax = axs[1]
    ax.plot(R.epoch, R["metrics/mAP50(B)"], color=BLUE, lw=2, label="mAP50")
    ax.plot(R.epoch, R["metrics/mAP50-95(B)"], color=ORANGE, lw=2, label="mAP50-95")
    ax.scatter([R.epoch[best]], [R["metrics/mAP50(B)"][best]], s=60, color=BLUE, edgecolor=SURFACE, linewidth=2, zorder=3)
    ax.annotate(f"meilleur epoch : {int(R.epoch[best])}\nmAP50 val {R['metrics/mAP50(B)'][best]:.3f}",
                (R.epoch[best], R["metrics/mAP50(B)"][best]), textcoords="offset points", xytext=(-150, -78),
                fontsize=9, color=INK2, arrowprops=dict(arrowstyle="-", color=MUTED))
    ax.set_title(f"Score sur la validation ({int(split_tbl.loc['val', 'Images'])} images)"); ax.set_xlabel("Epoch"); ax.legend(frameon=False, labelcolor=INK2, loc="lower right")
    plt.show()
    train_note = dict(epochs=len(R), best=int(R.epoch[best]), best_map50=float(R["metrics/mAP50(B)"][best]))
    msg = f"{train_note['epochs']} epochs ; meilleur epoch : **{train_note['best']}** (mAP50 validation {train_note['best_map50']:.3f})."
    if train_note["best"] >= train_note["epochs"] - 1:
        msg += " Le meilleur epoch est le dernier : le score montait encore, donc l'entraînement n'a pas convergé."
    display(Markdown(msg))

## 4. Évaluation des 4 variantes
Chaque modèle est évalué sur le jeu de test : AP50 par classe et latence image par image. **Cette cellule prend 1 à 2 minutes.**

In [ ]:
def eval_full(path):
    sess = ort.InferenceSession(str(path), providers=["CPUExecutionProvider"])
    inp = sess.get_inputs()[0].name
    thr = np.linspace(0.5, 0.95, 10)
    store = {c: {"tp": [], "conf": [], "n": 0} for c in range(NC)}
    times = []
    for f in P.TEST:
        x, shape, r, left, top = P.preprocess(f)
        t = time.perf_counter()
        out = sess.run(None, {inp: x})[0].astype(np.float32)
        times.append((time.perf_counter() - t) * 1000)
        dets = P.decode(out, shape, r, left, top)
        gt = P.load_gt(f, shape)
        for c in range(NC):
            gb = np.array([g[0] for g in gt if g[1] == c]).reshape(-1, 4)
            store[c]["n"] += len(gb)
            pd_ = sorted([d for d in dets if d[2] == c], key=lambda d: -d[1])
            tp = np.zeros((len(pd_), len(thr)))
            used = np.zeros((len(gb), len(thr)), bool)
            for k, (b, s, _) in enumerate(pd_):
                if len(gb) == 0:
                    break
                ious = P.iou_one(b, gb)
                for j, t_ in enumerate(thr):
                    cand = [(ious[q], q) for q in range(len(gb)) if not used[q, j] and ious[q] >= t_]
                    if cand:
                        q = max(cand)[1]; used[q, j] = True; tp[k, j] = 1
            store[c]["tp"].append(tp); store[c]["conf"].append(np.array([d[1] for d in pd_]))
    ap50 = np.full(NC, np.nan)
    for c in range(NC):
        if store[c]["n"]:
            ap50[c] = P.ap(np.concatenate(store[c]["tp"])[:, 0], np.concatenate(store[c]["conf"]), store[c]["n"])
    return ap50, np.array(times[3:])


variants = ["fp32", "fp16", "int8_dynamic", "int8_static"]
AP, LAT = {}, {}
for v in variants:
    AP[v], LAT[v] = eval_full(Path(f"models/plantdoc_{v}.onnx"))
    print(f"{VARIANT_LABEL[v]:15s} mAP50 {np.nanmean(AP[v]):.3f} | latence médiane {np.median(LAT[v]):.1f} ms")

## 5. Les performances
### Tableau de synthèse
Précision, taille et latence viennent de `results/benchmark_plantdoc.csv` (mesure du pipeline) ; les écarts sont calculés par rapport au FP32.

In [ ]:
bench = pd.read_csv("results/benchmark_plantdoc.csv", index_col="model")
base = bench.loc["fp32"]
perf_tbl = pd.DataFrame({
    "mAP50": bench.mAP50, "mAP50-95": bench.mAP50_95, "Taille (Mo)": bench.size_mb.round(2),
    "Latence médiane (ms)": bench.latency_ms_median.round(1),
    "Δ mAP50 vs FP32 (points)": ((bench.mAP50 - base.mAP50) * 100).round(1),
    "Taille ÷": (base.size_mb / bench.size_mb).round(1),
    "Vitesse ×": (base.latency_ms_median / bench.latency_ms_median).round(2),
}).loc[variants]
perf_tbl.index = [VARIANT_LABEL[v] for v in variants]
perf_tbl

### Précision, taille, latence

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(13, 3.9))
for ax, (col, title, fmt) in zip(axs, [("mAP50", "Précision (mAP50, test)", "{:.3f}"),
                                       ("size_mb", "Taille du fichier (Mo)", "{:.1f}"),
                                       ("latency_ms_median", "Latence médiane (ms, PC)", "{:.1f}")]):
    vals = [bench.loc[v, col] for v in variants]
    ax.bar(range(4), vals, color=[VARIANT_COLOR[v] for v in variants], width=0.62)
    for k, val in enumerate(vals):
        ax.text(k, val, fmt.format(val), ha="center", va="bottom", fontsize=9, color=INK2)
    ax.set_xticks(range(4)); ax.set_xticklabels(["FP32", "FP16", "INT8\ndyn.", "INT8\nstat."], fontsize=9)
    ax.set_title(title); ax.grid(axis="x", visible=False); ax.set_ylim(0, max(vals) * 1.15)
plt.show()

### Compromis précision / vitesse

In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 4.6))
for v in variants:
    ax.scatter(bench.loc[v, "latency_ms_median"], bench.loc[v, "mAP50"], s=40 + bench.loc[v, "size_mb"] * 14,
               color=VARIANT_COLOR[v], edgecolor=SURFACE, linewidth=2, zorder=3)
    ax.annotate(f"{VARIANT_LABEL[v]}\n{bench.loc[v, 'size_mb']:.1f} Mo", (bench.loc[v, "latency_ms_median"], bench.loc[v, "mAP50"]),
                textcoords="offset points", xytext=(12, 6), fontsize=9, color=INK2)
ax.set_xlabel("Latence médiane (ms), plus bas = mieux"); ax.set_ylabel("mAP50, plus haut = mieux")
ax.set_xlim(0, bench.latency_ms_median.max() * 1.3); ax.set_ylim(bench.mAP50.min() - 0.02, bench.mAP50.max() + 0.02)
ax.set_title("Compromis précision / vitesse (taille de la bulle = taille du fichier)")
plt.show()

### Régularité de la vitesse
Distribution des latences mesurées dans ce notebook (le PC n'étant pas isolé, les médianes peuvent différer un peu du tableau).

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.6))
bp = ax.boxplot([LAT[v] for v in variants], orientation="horizontal", widths=0.55, patch_artist=True, showfliers=False,
                medianprops=dict(color=INK, lw=2), whiskerprops=dict(color=MUTED), capprops=dict(color=MUTED))
for patch, v in zip(bp["boxes"], variants):
    patch.set_facecolor(VARIANT_COLOR[v]); patch.set_edgecolor(SURFACE)
ax.set_yticks(range(1, 5)); ax.set_yticklabels([VARIANT_LABEL[v] for v in variants], color=INK2)
ax.invert_yaxis(); ax.grid(axis="y", visible=False)
for k, v in enumerate(variants, start=1):
    ax.text(np.percentile(LAT[v], 75) + 0.6, k, f"médiane {np.median(LAT[v]):.1f} ms", va="center", fontsize=8.5, color=INK2)
ax.set_xlabel("Temps d'inférence par image (ms)"); ax.set_title("Étendue des latences par variante")
plt.show()

## 6. Où le modèle se trompe
### Précision par classe : FP32 contre INT8 statique

In [ ]:
ap32, ap8 = AP["fp32"], AP["int8_static"]
valid = [c for c in range(NC) if not np.isnan(ap32[c])]
ordr = sorted(valid, key=lambda c: ap32[c])
fig, ax = plt.subplots(figsize=(8.5, 8.2))
for k, c in enumerate(ordr):
    ax.plot([ap32[c], ap8[c]], [k, k], color=AXIS, lw=2, zorder=1)
ax.scatter([ap32[c] for c in ordr], range(len(ordr)), s=48, color=MUTED, edgecolor=SURFACE, linewidth=1.5, zorder=3, label="FP32")
ax.scatter([ap8[c] for c in ordr], range(len(ordr)), s=48, color=BLUE, edgecolor=SURFACE, linewidth=1.5, zorder=4, label="INT8 statique")
ax.set_yticks(range(len(ordr))); ax.set_yticklabels([f"{names[c]} (n={cls_counts.test[c]})" for c in ordr], fontsize=8.5, color=INK2)
ax.set_xlabel("AP50 sur le test (n = boîtes de test de la classe)"); ax.grid(axis="y", visible=False)
ax.legend(frameon=False, loc="lower right", labelcolor=INK2)
ax.set_title("Précision par classe : FP32 contre INT8 statique")
plt.show()

### La précision dépend-elle du nombre d'exemples d'entraînement ?

In [ ]:
xs = np.array([cls_counts.train[c] for c in valid]); ys = np.array([ap8[c] for c in valid])
rk = lambda a: pd.Series(a).rank().to_numpy()
rho = float(np.corrcoef(rk(xs), rk(ys))[0, 1])
fig, ax = plt.subplots(figsize=(7.2, 4.6))
ax.scatter(xs, ys, s=56, color=BLUE, edgecolor=SURFACE, linewidth=1.5, zorder=3)
for c, x, y in zip(valid, xs, ys):
    if y < 0.25 or y > 0.8 or x > 500:
        ax.annotate(names[c], (x, y), textcoords="offset points", xytext=(6, 4), fontsize=8, color=INK2)
ax.set_xscale("log"); ax.set_xlabel("Boîtes d'entraînement de la classe (échelle log)"); ax.set_ylabel("AP50 (INT8 statique, test)")
ax.set_title(f"AP50 selon le nombre d'exemples (corrélation de rang {rho:.2f})")
plt.show()

## 7. Chiffres clés et lecture

In [ ]:
tot = len(B)
weak = sorted(valid, key=lambda c: ap8[c])[:5]
drop = sorted(valid, key=lambda c: ap8[c] - ap32[c])[:3]
key = [
    ("Images", f"{len(I)} (train {split_tbl.loc['train', 'Images']:.0f}, val {split_tbl.loc['val', 'Images']:.0f}, test {split_tbl.loc['test', 'Images']:.0f})"),
    ("Boîtes annotées", f"{tot} ({tot / len(I):.1f} par image)"),
    ("Classes", f"{NC} ({sum(healthy)} saines, {NC - sum(healthy)} maladies ou parasites)"),
    ("Boîtes sur feuilles saines", f"{(B.type == 'saine').mean() * 100:.0f} %"),
    ("Déséquilibre", f"{int(cls_counts.total.max())} boîtes ({names[int(cls_counts.total.idxmax())]}) contre {int(cls_counts.total.min())} ; rapport {ratio:.0f}:1"),
    ("Surface d'une boîte", f"médiane {B.area.median() * 100:.0f} % de l'image"),
    ("Entraînement", (f"{train_note['epochs']} epochs, meilleur epoch {train_note['best']}, mAP50 validation {train_note['best_map50']:.3f}" if train_note else "journal introuvable")),
    ("Modèle déployé (INT8 statique)", f"mAP50 {bench.loc['int8_static', 'mAP50']:.3f}, {bench.loc['int8_static', 'size_mb']:.1f} Mo, {bench.loc['int8_static', 'latency_ms_median']:.1f} ms"),
    ("Gain INT8 statique vs FP32", f"taille ÷{perf_tbl.iloc[3]['Taille ÷']}, vitesse ×{perf_tbl.iloc[3]['Vitesse ×']}, mAP50 {perf_tbl.iloc[3]['Δ mAP50 vs FP32 (points)']:+.1f} point"),
    ("Classes les plus faibles (INT8 stat.)", ", ".join(f"{names[c]} ({ap8[c]:.2f})" for c in weak)),
    ("Plus forte baisse FP32 → INT8 stat.", ", ".join(f"{names[c]} ({(ap8[c] - ap32[c]) * 100:+.0f} pts)" for c in drop)),
    ("Précision vs exemples d'entraînement", f"corrélation de rang (Spearman) {rho:.2f}"),
]
pd.DataFrame(key, columns=["Indicateur", "Valeur"])

In [ ]:
display(Markdown(f'''
**Lecture**
- L'**INT8 statique** est la variante la plus rapide ({bench.loc['int8_static', 'latency_ms_median']:.1f} ms) et l'une des deux plus légères, pour {abs(perf_tbl.iloc[3]['Δ mAP50 vs FP32 (points)']):.1f} point de mAP50 perdu : c'est le meilleur compromis.
- L'**INT8 dynamique** est la plus petite mais la plus lente ; le **FP16** divise la taille par 2 sans gain de vitesse sur CPU.
- Un écart de moins de 1 point de mAP50 entre variantes n'est pas significatif sur ce petit jeu de test.
- Les classes les plus faibles sont : {', '.join(names[c] for c in weak)}. La corrélation entre le nombre d'exemples et l'AP50 est de {rho:.2f}.
- La quantization fait varier l'AP50 par classe dans les deux sens ({names[drop[0]]} : {(ap8[drop[0]] - ap32[drop[0]]) * 100:+.0f} points, sur {int(cls_counts.test[drop[0]])} boîtes de test) : avec si peu d'exemples par classe, ce sont des fluctuations de mesure.
'''))

## 8. Tableau complet par classe

In [ ]:
cls_out = pd.DataFrame({
    "Classe": names, "Type": ["saine" if h else "malade" for h in healthy],
    "Train": cls_counts.train, "Val": cls_counts.val, "Test": cls_counts.test, "Total": cls_counts.total,
    "AP50 FP32": [("" if np.isnan(ap32[c]) else round(float(ap32[c]), 2)) for c in range(NC)],
    "AP50 INT8 stat.": [("" if np.isnan(ap8[c]) else round(float(ap8[c]), 2)) for c in range(NC)],
}).sort_values("Total", ascending=False).reset_index(drop=True)
cls_out